In [1]:
import pandas as pd
from phd_helpers.paths import get_info_df, get_mesh, get_subject_stl_path, avg_edge_length
from pathlib import Path
import pyvista as pv
import subprocess
import numpy as np

## Full run up to cartilage for up to date chosen params 06/04/2026

In [12]:
path_MeshPipeline_main = '../../../../MeshPipeline/main.py'
subprocess.run(["python", path_MeshPipeline_main])


Updating parameters.json
	Wrote /Users/maro/Library/CloudStorage/OneDrive-UniversityofLeeds/PhD-wrist/WorkPackages/WorkPackages/TMCJ-Contact/Computational/MeshPipeline/set_parameters/parameters.json


Full parameter file saved to outputs/subs_okJustified/params/full_params.json

SUBJECT: 14548R
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 5.915s - ok
		STEP: cartilage
			RUN ID: -0-0
			Runtime: 6.199s - ok
	BONES: mc1-tpm
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 6.397s - ok
		STEP: cartilage
			RUN ID: -0-0
			Runtime: 8.586s - ok

SUBJECT: 14613R
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 6.476s - ok
		STEP: cartilage
			RUN ID: -0-0
			Runtime: 9.447s - ok
	BONES: mc1-tpm
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 6.500s - ok
		STEP: cartilage
			RUN ID: -0-0
			Runtime: 14.671s - ok

SUBJECT: 14685R
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 7.851s - ok
		STEP: cartilage
			RUN ID: -0-0
			Runtime: 7.857s - ok
	BONES: mc1-tpm
		STEP: 2Dmesh
		

CompletedProcess(args=['python', '../../../../MeshPipeline/main.py'], returncode=0)

In [6]:
def cartilage_h_A(mesh2d, min_h):
    """"return cartilage inner region height and proportion of inner area below min height"""

    bone_shell = mesh2d.extract_cells(mesh2d['region_id']==2, invert=True).extract_surface(algorithm=None)
    inner_cart = mesh2d.extract_cells(mesh2d['inner_cells']==1).extract_surface(algorithm=None)
    cart_height = inner_cart.cell_centers().compute_implicit_distance(bone_shell)['implicit_distance']

    As = inner_cart.compute_cell_sizes()['Area']
    A_inner = np.sum(As)

    below_mask = cart_height < min_h # cell centers below min height
    A_below = np.sum(As[below_mask]) / A_inner # proportion of area below min height

    return cart_height, A_below


In [17]:
n_tets, min_size = 3, 0.06
min_h = n_tets * min_size

#root_dir = Path('outputs/subs_okJustified')
root_dir = Path('../../../../MeshPipeline/outputs/ParamOptimisation/fullRuns')
mesh2d_paths = list(root_dir.glob('**/bone_cartilage_mesh*.vtp'))
mesh3d_paths = list(root_dir.glob('**/mesh*.vtu'))
data = []
for mesh2d_path, mesh3d_path in zip(mesh2d_paths, mesh3d_paths):
    mesh2d = pv.read(mesh2d_path)
    mesh3d = pv.read(mesh3d_path)
    cart_height, A_below = cartilage_h_A(mesh2d, min_h)

    bone, _ = mesh2d_path.parent.parent.name.split('-')
    sub = mesh2d_path.parents[2].name
    subject, side = sub[:-1], sub[-1]
    L = avg_edge_length(get_mesh(get_subject_stl_path(subject, side), 'tpm'))

    data.append({
        'subject': sub,
        'bone': bone,
        'n_cells': mesh3d.extract_cells_by_type(10).n_cells,
        'h_mean': cart_height.mean(),
        'A_below %': A_below*100,
        'h_min': cart_height.min(),
        'h_std': np.std(cart_height),
        'L_orig': L
    })

In [18]:
bone = 'tpm'

df = pd.DataFrame(data).sort_values('h_mean')
df = df[df['bone']==bone]
#runs = pd.read_json(root_dir / 'reports/runtimes.jsonl', lines=True)
#runs = runs[runs['step']=='3Dmesh'][['subject', 'runtime']]
vols = get_info_df()
vols['subject'] = [x+y for x, y in zip(vols['subject'].astype(str), vols['side'])]
vols = vols[vols['group']=='CMC'][['subject', 'tpm_volume']]
#a = pd.merge(df, runs, on='subject')
metric_df = pd.merge(df, vols, on='subject').sort_values('h_mean')
metric_df

,subject,bone,n_cells,h_mean,A_below %,h_min,h_std,L_orig,tpm_volume
0,50021R,tpm,581410,0.264730,31.142556,0.065893,0.120868,0.369239,1962.99
1,14819R,tpm,574988,0.266487,33.755641,0.058198,0.137017,0.440327,1998.23
2,14874R,tpm,473821,0.273687,36.082743,0.059451,0.142339,0.441794,1437.32
3,50029R,tpm,409632,0.299712,19.113186,0.056365,0.124875,0.368228,1639.48
4,15283R,tpm,503820,0.329545,9.360993,0.093736,0.101766,0.513917,2344.49
5,50045R,tpm,457281,0.340526,3.183827,0.154803,0.118881,0.366641,2682.14
6,15294R,tpm,380562,0.345378,14.344007,0.059115,0.140946,0.444981,1981.79
7,50014R,tpm,371155,0.357814,0.623002,0.168335,0.122832,0.364354,1850.40
8,14726R,tpm,611066,0.358475,4.306510,0.140836,0.121613,0.442901,3500.61
9,50000R,tpm,230803,0.369456,0.143156,0.175002,0.077913,0.366141,1305.55
